# LeetCode 3816 — Lexicographically Smallest String After Deleting Duplicate Characters bằng Python

- Link: https://leetcode.com/problems/lexicographically-smallest-string-after-deleting-duplicate-characters/
- Độ khó: **Hard**
- Chủ đề: Chuỗi, tham lam, ngăn xếp đơn điệu (monotonic stack), dãy con nhỏ nhất theo từ điển, quy tắc "tiền tố nhỏ hơn"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def lexSmallestAfterDeletion(self, s: str) -> str:
```

## 1. Mục tiêu học

1. Biến bài toán "thao tác xóa lặp lại" thành mô tả tĩnh: kết quả là **một dãy con** của `s` **giữ lại ít nhất một** lần xuất hiện của mỗi chữ cái.
2. Nhớ quy tắc so sánh từ điển: một **tiền tố thật sự** nhỏ hơn chuỗi dài hơn (`"ab" < "abc"`) — nên khi được phép thì **xóa ở cuối** luôn có lợi.
3. Cài đặt **ngăn xếp đơn điệu**: bỏ đỉnh lớn hơn ký tự hiện tại nếu chữ cái đó còn một bản khác.
4. Viết lời giải tham lam "chọn ký tự nhỏ nhất khả thi" để đối chiếu.
5. Kiểm thử với BFS vét cạn mọi chuỗi đạt được.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho chuỗi `s` (chữ thường). Thao tác: chọn một chữ cái đang xuất hiện **ít nhất hai lần** trong chuỗi hiện tại và **xóa một** lần xuất hiện bất kỳ của nó. Làm bao nhiêu lần cũng được. Trả về chuỗi **nhỏ nhất theo từ điển** có thể tạo ra.

**Ví dụ 1:** `s = "aaccb"` → `"aacb"`. Các chuỗi đạt được: `"acb"`, `"aacb"`, `"accb"`, `"aaccb"`; nhỏ nhất là `"aacb"`.

**Ví dụ 2:** `s = "z"` → `"z"`.

**Ràng buộc:** `1 ≤ |s| ≤ 10^5`, chữ thường.

In [1]:
from collections import Counter, deque
import random, time

examples = [("aaccb", "aacb"), ("z", "z")]
for s, exp in examples:
    print(repr(s), "-> kỳ vọng", repr(exp))

'aaccb' -> kỳ vọng 'aacb'
'z' -> kỳ vọng 'z'


## 3. Kiến thức nền

### 3.1. So sánh chuỗi theo từ điển trong Python

So từng ký tự từ trái; ký tự khác nhau đầu tiên quyết định. Nếu một chuỗi là **tiền tố** của chuỗi kia thì chuỗi ngắn hơn nhỏ hơn.

In [2]:
print("aacb" < "acb", "ab" < "abc", "b" < "abc", min(["acb", "aacb", "accb", "aaccb"]))

True True False aacb


### 3.2. Ngăn xếp đơn điệu (mẫu "xóa để được dãy nhỏ nhất")

Duyệt từng ký tự; trong khi đỉnh ngăn xếp **lớn hơn** ký tự hiện tại **và** được phép bỏ đỉnh, thì bỏ. Mẫu kinh điển: *Remove K Digits* (LC 402), *Remove Duplicate Letters* (LC 316).

In [3]:
def remove_k_digits(num, k):
    st = []
    for c in num:
        while st and k and st[-1] > c:
            st.pop(); k -= 1
        st.append(c)
    st = st[:len(st) - k] if k else st
    return "".join(st).lstrip("0") or "0"

print(remove_k_digits("1432219", 3))

1219


### 3.3. `Counter` theo dõi số bản còn lại

`cnt[c]` = số lần chữ `c` có trong **chuỗi hiện tại** (các ký tự đã giữ trong ngăn xếp + các ký tự chưa duyệt). Bỏ một bản ⇒ `cnt[c] −= 1`. Chỉ được bỏ khi `cnt[c] > 1`.

In [4]:
cnt = Counter("aaccb")
print(cnt, "| được xóa một 'c'?", cnt['c'] > 1, "| được xóa 'b'?", cnt['b'] > 1)

Counter({'a': 2, 'c': 2, 'b': 1}) | được xóa một 'c'? True | được xóa 'b'? False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

**Mô tả tĩnh các chuỗi đạt được.** Mỗi thao tác xóa một ký tự mà chữ cái của nó vẫn còn bản khác ⇒ mọi chữ cái của `s` luôn còn ít nhất một bản. Ngược lại, **mọi dãy con** của `s` giữ lại ≥ 1 bản của mỗi chữ cái đều đạt được (xóa lần lượt các ký tự không giữ; lúc xóa, chữ đó vẫn còn bản được giữ).

Vậy bài toán: **dãy con nhỏ nhất theo từ điển** của `s` chứa đủ mọi chữ cái (mỗi chữ ≥ 1 lần, được phép nhiều lần).

Tính tay `s = "aaccb"`: phải giữ `a`, `c`, `b`. Dãy con bắt đầu bằng `a`, rồi `a` (giữ thêm `a` thứ hai vì `"aa…" < "ac…"`), rồi phải có `c` và `b` — `b` nằm cuối nên cần một `c` trước nó ⇒ `"aacb"`. Không thể là `"aab"` vì thiếu `c`.

**Quy tắc then chốt:** khi một chữ cái lớn đứng trước một chữ cái nhỏ hơn, và chữ lớn đó còn bản khác, xóa nó làm chuỗi nhỏ hơn. Còn ở **cuối** chuỗi, mọi bản thừa đều nên xóa (rút ngắn thành tiền tố nhỏ hơn).

## 5. Cách 1 — Brute Force: BFS mọi chuỗi đạt được

Bắt đầu từ `s`, áp dụng mọi thao tác hợp lệ, thu tất cả chuỗi đạt được, lấy `min`. Số chuỗi có thể lên tới `2^n` — chỉ dùng cho chuỗi ngắn.

In [5]:
def brute(s):
    seen = {s}
    dq = deque([s])
    while dq:
        t = dq.popleft()
        cnt = Counter(t)
        for i, c in enumerate(t):
            if cnt[c] >= 2:
                u = t[:i] + t[i + 1:]
                if u not in seen:
                    seen.add(u); dq.append(u)
    return min(seen)

for s, exp in examples:
    print(repr(brute(s)), "kỳ vọng", repr(exp))
print("brute('aaccb') =", brute("aaccb"))

'aacb' kỳ vọng 'aacb'
'z' kỳ vọng 'z'
brute('aaccb') = aacb


**Nút thắt:** số trạng thái hàm mũ. Cần tham lam dựa trên mô tả tĩnh ở mục 4.

## 6. Ý tưởng thuật toán / bước đột phá

**Ngăn xếp đơn điệu với điều kiện "còn bản khác".** Duyệt `s` từ trái; `cnt` là số bản của mỗi chữ trong "chuỗi hiện tại" (ngăn xếp + phần chưa duyệt):

- trong khi đỉnh `top > c` và `cnt[top] > 1`: bỏ đỉnh (`cnt[top] −= 1`) — chữ `top` vẫn còn bản ở dưới ngăn xếp hoặc phía sau;
- đẩy `c`.

**Sau khi duyệt xong:** trong khi đỉnh còn bản khác (`cnt[top] > 1`), bỏ đỉnh — rút ngắn thành tiền tố (nhỏ hơn).

Khác với LC 316 (mỗi chữ **đúng một** lần), ở đây chữ cái **được** giữ nhiều lần nếu có lợi (ví dụ `"aa"` ở đầu `"aacb"`), và không cần "đã có trong ngăn xếp thì bỏ qua".

## 7. Các cách tiếp theo

- **Cách 2:** tham lam **chọn ký tự** từng bước: ở mỗi bước, chọn ký tự nhỏ nhất ở vị trí sớm nhất sao cho các chữ **chưa có** vẫn còn xuất hiện phía sau; dừng ngay khi đã đủ mọi chữ — `O(26 · n)`.
- **Cách 3:** ngăn xếp đơn điệu (lời giải nộp chính) — `O(n)`.

### 7.1. Cách 2 — Tham lam chọn ký tự tiếp theo

**Giải thích:**

1. `nxt[i][c]` = vị trí xuất hiện đầu tiên của `c` tại hoặc sau `i`; `last[c]` = vị trí cuối cùng của `c`.
2. Đang ở vị trí `pos`, tập chữ **cần** `need`. Nếu `need` rỗng ⇒ dừng (tiền tố nhỏ nhất).
3. Thử `c = 'a'..'z'`: `j = nxt[pos][c]`; khả thi nếu mọi chữ `d ∈ need`, `d ≠ c` có `last[d] > j`. Chọn `c` khả thi đầu tiên, thêm vào kết quả, `pos = j + 1`, bỏ `c` khỏi `need`.

In [6]:
def greedy_pick(s):
    n = len(s)
    nxt = [[n] * 26 for _ in range(n + 1)]
    for i in range(n - 1, -1, -1):
        nxt[i] = nxt[i + 1][:]
        nxt[i][ord(s[i]) - 97] = i
    last = {}
    for i, ch in enumerate(s): last[ch] = i
    need = set(s)
    res, pos = [], 0
    while need:
        for c in range(26):
            j = nxt[pos][c]
            if j == n: continue
            ch = chr(97 + c)
            if all(last[d] > j for d in need if d != ch):
                res.append(ch); need.discard(ch); pos = j + 1
                break
    return "".join(res)

for s, exp in examples:
    print(repr(greedy_pick(s)), "kỳ vọng", repr(exp))

'aacb' kỳ vọng 'aacb'
'z' kỳ vọng 'z'


**Độ phức tạp:** mỗi bước thử ≤ 26 ký tự, mỗi lần kiểm tra ≤ 26 chữ ⇒ `O(26² · độ dài kết quả)` + bảng `nxt` `O(26 n)` bộ nhớ.

**Vì sao được chọn chữ đã có?** Nếu chữ `c` nhỏ nhất khả thi đã có trong kết quả, thêm nó vẫn tốt hơn mọi lựa chọn khác (cùng tiền tố, ký tự tiếp theo nhỏ hơn) — đó là lý do `"aacb"` có hai `a`.

### 7.2. Cách 3 — Ngăn xếp đơn điệu (lời giải chính)

**Giải thích từng đoạn:**

1. `cnt = Counter(s)`.
2. Với mỗi `c`: `while st and st[-1] > c and cnt[st[-1]] > 1: cnt[st.pop()] −= 1`; rồi `st.append(c)`.
3. Cuối: `while st and cnt[st[-1]] > 1: cnt[st.pop()] −= 1`.

In [7]:
def mono_stack(s):
    cnt = Counter(s)
    st = []
    for c in s:
        while st and st[-1] > c and cnt[st[-1]] > 1:
            cnt[st.pop()] -= 1
        st.append(c)
    while st and cnt[st[-1]] > 1:
        cnt[st.pop()] -= 1
    return "".join(st)

for s, exp in examples:
    print(repr(mono_stack(s)), "kỳ vọng", repr(exp))

'aacb' kỳ vọng 'aacb'
'z' kỳ vọng 'z'


**Độ phức tạp Cách 3:** mỗi ký tự vào/ra ngăn xếp tối đa một lần ⇒ `O(n)`; bộ nhớ `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 trên `"aaccb"`.

| bước | c | ngăn xếp trước | bỏ đỉnh? | ngăn xếp sau | cnt |
|---|---|---|---|---|---|
| 1 | a | [] | – | [a] | a2 c2 b1 |
| 2 | a | [a] | `a > a`? không | [a, a] | |
| 3 | c | [a, a] | không | [a, a, c] | |
| 4 | c | [a, a, c] | `c > c`? không | [a, a, c, c] | |
| 5 | b | [a, a, c, c] | `c > b`, cnt[c] = 2 > 1 ⇒ bỏ; đỉnh `c`, cnt[c] = 1 ⇒ dừng | [a, a, c, b] | c1 |
| cuối | – | [a, a, c, b] | đỉnh `b`, cnt = 1 ⇒ dừng | [a, a, c, b] | |

In [8]:
def trace(s):
    cnt = Counter(s); st = []
    for c in s:
        popped = []
        while st and st[-1] > c and cnt[st[-1]] > 1:
            x = st.pop(); cnt[x] -= 1; popped.append(x)
        st.append(c)
        print(f"c={c}: bỏ {popped} -> ngăn xếp {''.join(st)} | cnt={dict(cnt)}")
    tail = []
    while st and cnt[st[-1]] > 1:
        x = st.pop(); cnt[x] -= 1; tail.append(x)
    print("bỏ ở cuối:", tail, "-> kết quả", "".join(st))

trace("aaccb")

c=a: bỏ [] -> ngăn xếp a | cnt={'a': 2, 'c': 2, 'b': 1}
c=a: bỏ [] -> ngăn xếp aa | cnt={'a': 2, 'c': 2, 'b': 1}
c=c: bỏ [] -> ngăn xếp aac | cnt={'a': 2, 'c': 2, 'b': 1}
c=c: bỏ [] -> ngăn xếp aacc | cnt={'a': 2, 'c': 2, 'b': 1}
c=b: bỏ ['c'] -> ngăn xếp aacb | cnt={'a': 2, 'c': 1, 'b': 1}
bỏ ở cuối: [] -> kết quả aacb


In [9]:
trace("cbacdcbc")
print("vét cạn:", brute("cbacdcbc"))

c=c: bỏ [] -> ngăn xếp c | cnt={'c': 4, 'b': 2, 'a': 1, 'd': 1}
c=b: bỏ ['c'] -> ngăn xếp b | cnt={'c': 3, 'b': 2, 'a': 1, 'd': 1}
c=a: bỏ ['b'] -> ngăn xếp a | cnt={'c': 3, 'b': 1, 'a': 1, 'd': 1}
c=c: bỏ [] -> ngăn xếp ac | cnt={'c': 3, 'b': 1, 'a': 1, 'd': 1}
c=d: bỏ [] -> ngăn xếp acd | cnt={'c': 3, 'b': 1, 'a': 1, 'd': 1}
c=c: bỏ [] -> ngăn xếp acdc | cnt={'c': 3, 'b': 1, 'a': 1, 'd': 1}
c=b: bỏ ['c'] -> ngăn xếp acdb | cnt={'c': 2, 'b': 1, 'a': 1, 'd': 1}
c=c: bỏ [] -> ngăn xếp acdbc | cnt={'c': 2, 'b': 1, 'a': 1, 'd': 1}
bỏ ở cuối: ['c'] -> kết quả acdb
vét cạn: acdb


Ở `"cbacdcbc"`: kết quả `"acdb"` (giống LC 316 vì mọi chữ đều nên giữ một lần). Bước cuối xóa `"c"` thừa ở đuôi — rút ngắn thành tiền tố nhỏ hơn.

## 9. Code nộp LeetCode

In [10]:
from collections import Counter

class Solution:
    def lexSmallestAfterDeletion(self, s: str) -> str:
        cnt = Counter(s)          # số bản trong "chuỗi hiện tại" (ngăn xếp + phần chưa duyệt)
        st = []
        for c in s:
            while st and st[-1] > c and cnt[st[-1]] > 1:
                cnt[st.pop()] -= 1
            st.append(c)
        while st and cnt[st[-1]] > 1:   # bản thừa ở cuối: xóa để thành tiền tố nhỏ hơn
            cnt[st.pop()] -= 1
        return "".join(st)

sol = Solution()
for s, exp in examples:
    got = sol.lexSmallestAfterDeletion(s)
    print(repr(s), "->", repr(got), "✔" if got == exp else "✘")

'aaccb' -> 'aacb' ✔
'z' -> 'z' ✔


## 10. Vì sao thuật toán đúng

1. **Mô tả tĩnh:** tập chuỗi đạt được = tập dãy con của `s` chứa mọi chữ cái của `s` (đã chứng minh ở mục 4).
2. **Bất biến của ngăn xếp:** sau khi xử lý tiền tố `s[0..i]`, ngăn xếp là dãy con nhỏ nhất theo từ điển của tiền tố đó **trong số các dãy con mà khi nối với phần còn lại `s[i+1..]` vẫn có thể chứa đủ chữ cái** — với điều kiện ngầm là phần còn lại được giữ nguyên (sẽ xử lý tiếp). Bỏ đỉnh `top > c` khi `cnt[top] > 1` làm vị trí đó nhận `c` nhỏ hơn mà không mất chữ `top` (còn bản khác trong ngăn xếp hoặc phía sau).
3. **Không bỏ khi không nên:** nếu `top ≤ c`, bỏ `top` làm ký tự tại vị trí đó lớn hơn hoặc bằng ⇒ không tốt hơn; nếu `cnt[top] = 1`, không được bỏ.
4. **Đuôi:** sau khi duyệt hết, nếu ký tự cuối của ngăn xếp còn bản khác (ở phía dưới) thì xóa nó cho ta một **tiền tố** thật sự — luôn nhỏ hơn. Lặp tới khi ký tự cuối là bản duy nhất của chữ đó (xóa sẽ mất chữ).
5. Đối chiếu với BFS vét cạn ở mục 12 xác nhận trên hàng nghìn chuỗi ngẫu nhiên.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS mọi chuỗi | `O(2^n · n)` | `O(2^n · n)` |
| 2. Tham lam chọn ký tự | `O(26 n + 26² · |kq|)` | `O(26 n)` |
| 3. Ngăn xếp đơn điệu | `O(n)` | `O(n)` |

## 12. Bộ test

In [11]:
def run_all(s, with_brute=True):
    res = [greedy_pick(s), mono_stack(s), Solution().lexSmallestAfterDeletion(s)]
    if with_brute:
        res.append(brute(s))
    return res

for s, exp in examples:
    assert all(v == exp for v in run_all(s))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng BFS)
cases = [("aaaa", "a"), ("ba", "ba"), ("abab", "aab"), ("bab", "ab"), ("bcabc", "abc"), ("zyxz", "yxz"), ("abacb", "aacb")]
for s, exp in cases:
    r = run_all(s)
    print(repr(s), "->", r)
    assert all(v == r[-1] == exp for v in r)

'aaaa' -> ['a', 'a', 'a', 'a']
'ba' -> ['ba', 'ba', 'ba', 'ba']
'abab' -> ['aab', 'aab', 'aab', 'aab']
'bab' -> ['ab', 'ab', 'ab', 'ab']
'bcabc' -> ['abc', 'abc', 'abc', 'abc']
'zyxz' -> ['yxz', 'yxz', 'yxz', 'yxz']
'abacb' -> ['aacb', 'aacb', 'aacb', 'aacb']


Một số kết quả đáng chú ý (đều khớp BFS):

- `"abab"` → `"aab"`, **không** phải `"ab"`: so sánh `"aab"` với `"ab"` ở ký tự thứ hai: `'a' < 'b'` ⇒ `"aab"` nhỏ hơn. Giữ thêm một `a` "chen" trước `b` có lợi hơn rút ngắn. (Ngăn xếp: gặp `a` thứ hai thì bỏ `b` ở đỉnh vì còn `b` phía sau.)
- `"aaaa"` → `"a"`: xóa ba bản thừa ở đuôi (tiền tố ngắn hơn).
- `"zyxz"` → `"yxz"`: `z` đầu có bản khác ở cuối nên bỏ được; `y`, `x` chỉ có một bản nên phải giữ.
- `"abacb"` → `"aacb"`: giống ví dụ 1.

In [13]:
# 12.3 phản ví dụ: dùng lời giải LC 316 (mỗi chữ đúng một lần) cho bài này là SAI
def lc316(s):
    last = {c: i for i, c in enumerate(s)}; st = []; inside = set()
    for i, c in enumerate(s):
        if c in inside: continue
        while st and st[-1] > c and last[st[-1]] > i:
            inside.discard(st.pop())
        st.append(c); inside.add(c)
    return "".join(st)

print("LC 316:", lc316("aaccb"), "| đúng:", Solution().lexSmallestAfterDeletion("aaccb"))

LC 316: acb | đúng: aacb


In [14]:
# 12.4 đối chiếu ngẫu nhiên với BFS
rng = random.Random(3816)
for _ in range(600):
    s = "".join(rng.choice("abc"[:rng.randint(1, 3)] + "d" * (rng.random() < 0.3)) for _ in range(rng.randint(1, 9)))
    r = run_all(s)
    assert len(set(r)) == 1, (s, r)
print("600 test ngẫu nhiên: OK")

600 test ngẫu nhiên: OK


In [15]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(9)
s = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(10**5))
t0 = time.time(); a = Solution().lexSmallestAfterDeletion(s); t1 = time.time()
print("độ dài kết quả:", len(a), "| khớp tham lam chọn ký tự:", a == greedy_pick(s), "| thời gian < 1s:", t1 - t0 < 1)
print("toàn 'z' trừ một 'a' ở cuối:", Solution().lexSmallestAfterDeletion("z" * 10**5 + "a"))

độ dài kết quả: 3878 | khớp tham lam chọn ký tự: True | thời gian < 1s: True


toàn 'z' trừ một 'a' ở cuối: za


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Áp dụng nguyên lời giải LC 316 (mỗi chữ đúng một lần)

Đã thấy ở 12.3: LC 316 cho `"acb"` nhưng đáp án là `"aacb"` — **giữ thêm** bản `a` có lợi vì `"aa…" < "ac…"`.

### Lỗi 2 — Quên xóa bản thừa ở **cuối**

In [16]:
def wrong_no_tail(s):
    cnt = Counter(s); st = []
    for c in s:
        while st and st[-1] > c and cnt[st[-1]] > 1:
            cnt[st.pop()] -= 1
        st.append(c)
    return "".join(st)                         # SAI: chưa bỏ đuôi thừa

print("sai:", wrong_no_tail("aaaa"), wrong_no_tail("abab"), "| đúng:", Solution().lexSmallestAfterDeletion("aaaa"), Solution().lexSmallestAfterDeletion("abab"))

sai: aaaa aab | đúng: a aab


### Lỗi 3 — Đếm `cnt` là "số bản **phía sau**" thay vì "số bản trong chuỗi hiện tại"

Nếu chỉ đếm phần chưa duyệt (giảm `cnt` khi duyệt qua), ta không cho phép xóa một bản khi bản còn lại nằm **trong ngăn xếp phía dưới** — bỏ lỡ cơ hội.

In [17]:
def wrong_future_only(s):
    rem = Counter(s); st = []
    for c in s:
        rem[c] -= 1                              # SAI: chỉ đếm phía sau
        while st and st[-1] > c and rem[st[-1]] > 0:
            st.pop()
        st.append(c)
    return "".join(st)

rng = random.Random(3)
for _ in range(5000):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 8)))
    if wrong_future_only(s) != Solution().lexSmallestAfterDeletion(s):
        print("phản ví dụ:", repr(s), "| sai:", wrong_future_only(s), "| đúng:", Solution().lexSmallestAfterDeletion(s)); break

phản ví dụ: 'ccab' | sai: ccab | đúng: cab


### Lỗi 4 — Bỏ đỉnh khi `top ≥ c` (kể cả bằng)

Bỏ một bản bằng `c` rồi đẩy `c` không thay đổi chuỗi ở vị trí đó nhưng làm mất một bản — có thể khiến bản cuối cùng của chữ này rơi vào vị trí kém hơn hoặc vi phạm điều kiện.

In [18]:
def wrong_ge(s):
    cnt = Counter(s); st = []
    for c in s:
        while st and st[-1] >= c and cnt[st[-1]] > 1:   # SAI: >=
            cnt[st.pop()] -= 1
        st.append(c)
    while st and cnt[st[-1]] > 1:
        cnt[st.pop()] -= 1
    return "".join(st)

rng = random.Random(4)
for _ in range(3000):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 8)))
    if wrong_ge(s) != Solution().lexSmallestAfterDeletion(s):
        print("phản ví dụ:", repr(s), "| sai:", wrong_ge(s), "| đúng:", Solution().lexSmallestAfterDeletion(s)); break
else:
    print("không tìm thấy phản ví dụ trong 3000 chuỗi — điều kiện >= ở đây tình cờ vẫn đúng, nhưng dùng > rõ ràng và an toàn hơn")

phản ví dụ: 'aaaabcba' | sai: abc | đúng: aaaabc


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Dãy con nhỏ nhất theo từ điển với ràng buộc giữ một số ký tự"** ⇒ ngăn xếp đơn điệu với điều kiện "được bỏ đỉnh" phù hợp ràng buộc.
- **Thao tác lặp lại** ⇒ mô tả tập kết quả đạt được (bất biến) trước khi tối ưu.
- **Tiền tố nhỏ hơn** ⇒ luôn cắt đuôi thừa nếu được phép.
- So sánh với các bài cùng họ: LC 316/1081 (mỗi chữ đúng một lần), LC 402 (xóa `k` chữ số), LC 1673 (dãy con độ dài `k` nhỏ nhất).

## 15. Biến thể của bài

1. **Mỗi chữ phải giữ đúng một lần**: LC 316.
2. **Mỗi chữ phải giữ ít nhất `t` lần**: điều kiện bỏ đỉnh `cnt[top] > t`, đuôi tương tự.
3. **Chuỗi lớn nhất theo từ điển**: đảo chiều so sánh — nhưng chú ý chuỗi dài hơn có cùng tiền tố thì **lớn** hơn, nên không cắt đuôi.
4. **Chỉ được thực hiện tối đa `k` thao tác**: thêm bộ đếm số lần xóa.

In [19]:
# Biến thể 2: mỗi chữ phải còn ít nhất t bản (chữ có ít hơn t bản ban đầu thì giữ nguyên tất cả)
def keep_at_least(s, t):
    cnt = Counter(s); need = {c: min(t, v) for c, v in cnt.items()}; st = []
    for c in s:
        while st and st[-1] > c and cnt[st[-1]] > need[st[-1]]:
            cnt[st.pop()] -= 1
        st.append(c)
    while st and cnt[st[-1]] > need[st[-1]]:
        cnt[st.pop()] -= 1
    return "".join(st)

def brute_at_least(s, t):
    from itertools import combinations
    full = Counter(s); best = None
    for r in range(len(s) + 1):
        for idx in combinations(range(len(s)), r):
            sub = "".join(s[i] for i in idx); c = Counter(sub)
            if all(c[x] >= min(t, full[x]) for x in full):
                best = sub if best is None or sub < best else best
    return best

rng = random.Random(15)
for _ in range(300):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 8))); t = rng.randint(1, 3)
    assert keep_at_least(s, t) == brute_at_least(s, t), (s, t)
print("Biến thể 2 khớp vét cạn; với t = 1 trùng bài gốc:", keep_at_least("aaccb", 1))

Biến thể 2 khớp vét cạn; với t = 1 trùng bài gốc: aacb


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | Mọi chuỗi đạt được | `O(2^n n)` | lớn | Kiểm thử |
| 2. Tham lam chọn ký tự | Ký tự nhỏ nhất khả thi | `O(26 n)` | `O(26 n)` | Dễ chứng minh |
| 3. Ngăn xếp đơn điệu | Bỏ đỉnh lớn còn bản khác | `O(n)` | `O(n)` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `lexSmallestAfterDeletion("bba")` = ?

**Bài 2 (tính tay).** `"cabca"` — trace ngăn xếp.

**Bài 3 (điền TODO).** Viết `reachable_check(s, t)` kiểm tra `t` có đạt được từ `s` không (dùng mô tả tĩnh).

**Bài 4 (tự viết lại).** Viết lại Cách 3 dùng mảng đếm 26 phần tử thay cho `Counter`.

**Bài 5 (mini challenge).** Biến thể "chuỗi **lớn nhất**" theo từ điển (cùng thao tác). Viết và kiểm chứng bằng BFS.

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def reachable_check(s, t):
    # TODO: t là dãy con của s và set(t) == set(s)
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def lex_largest(s):
    # TODO: ngăn xếp: bỏ đỉnh NHỎ hơn c nếu còn bản khác; KHÔNG cắt đuôi
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Phải giữ `b` và `a`. (2) `b` đầu có bản khác ⇒ bỏ khi gặp `a`? Đỉnh lúc gặp `a` là `b` (thứ hai) và `cnt[b] = 2` ⇒ bỏ; đỉnh tiếp `b`, `cnt = 1` ⇒ dừng. (3) `"ba"`.

**Bài 2.** (1) `c` có 2 bản, `a` có 2 bản. (2) Gặp `a` (vị trí 1): bỏ `c` (còn bản sau). (3) Kết quả `"abca"` → đuôi `a` thừa (còn `a` đầu) ⇒ `"abc"`.

**Bài 3.** (1) Kiểm tra dãy con bằng con trỏ. (2) So tập chữ cái. (3) Đối chiếu BFS.

**Bài 4.** (1) `cnt = [0]*26`. (2) Chỉ số `ord(c) − 97`. (3) Giữ nguyên logic.

**Bài 5.** (1) Đảo so sánh khi bỏ đỉnh. (2) Chuỗi dài hơn cùng tiền tố thì lớn hơn ⇒ không cắt đuôi. (3) BFS với `max`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().lexSmallestAfterDeletion("bba") == "ba" == brute("bba")
assert Solution().lexSmallestAfterDeletion("cabca") == brute("cabca")
print("Bài 1:", Solution().lexSmallestAfterDeletion("bba"), "| Bài 2:", Solution().lexSmallestAfterDeletion("cabca"))

Bài 1: ba | Bài 2: abc


In [23]:
# Bài 3
def reachable_check_sol(s, t):
    it = iter(s)
    return all(ch in it for ch in t) and set(t) == set(s)

def all_reachable(s):
    seen = {s}; dq = deque([s])
    while dq:
        x = dq.popleft(); cnt = Counter(x)
        for i, c in enumerate(x):
            if cnt[c] >= 2:
                u = x[:i] + x[i + 1:]
                if u not in seen: seen.add(u); dq.append(u)
    return seen

from itertools import combinations
for s in ("aaccb", "abab", "cbacdc"):
    R = all_reachable(s)
    subs = {"".join(s[i] for i in idx) for r in range(len(s) + 1) for idx in combinations(range(len(s)), r)}
    assert {t for t in subs if reachable_check_sol(s, t)} == R
print("Bài 3 OK: mô tả tĩnh khớp tập đạt được bằng BFS")

Bài 3 OK: mô tả tĩnh khớp tập đạt được bằng BFS


In [24]:
# Bài 4
def mono_stack_array(s):
    cnt = [0] * 26
    for c in s: cnt[ord(c) - 97] += 1
    st = []
    for c in s:
        while st and st[-1] > c and cnt[ord(st[-1]) - 97] > 1:
            cnt[ord(st.pop()) - 97] -= 1
        st.append(c)
    while st and cnt[ord(st[-1]) - 97] > 1:
        cnt[ord(st.pop()) - 97] -= 1
    return "".join(st)

rng = random.Random(4)
for _ in range(500):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 30)))
    assert mono_stack_array(s) == Solution().lexSmallestAfterDeletion(s)
print("Bài 4 OK")

Bài 4 OK


In [25]:
# Bài 5: chuỗi lớn nhất
def lex_largest_sol(s):
    cnt = Counter(s); st = []
    for c in s:
        while st and st[-1] < c and cnt[st[-1]] > 1:
            cnt[st.pop()] -= 1
        st.append(c)
    return "".join(st)

def brute_largest(s):
    return max(all_reachable(s))

rng = random.Random(5)
for _ in range(400):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 8)))
    assert lex_largest_sol(s) == brute_largest(s), s
print("Bài 5 OK, ví dụ:", lex_largest_sol("aaccb"))

Bài 5 OK, ví dụ: accb


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
cnt = Counter(s); st = []
for c in s:
    while st and st[-1] > c and cnt[st[-1]] > 1:
        cnt[st.pop()] -= 1
    st.append(c)
while st and cnt[st[-1]] > 1:      # cắt đuôi thừa
    cnt[st.pop()] -= 1
return "".join(st)
```

**Checklist:**

- [ ] `cnt` là số bản trong **chuỗi hiện tại** (ngăn xếp + phần chưa duyệt), không chỉ phần sau?
- [ ] Bỏ đỉnh khi `top > c` (ngặt) và `cnt[top] > 1`?
- [ ] Không bỏ qua chữ đã có trong ngăn xếp (khác LC 316)?
- [ ] Cắt đuôi thừa sau vòng lặp?
- [ ] Chuỗi một ký tự trả về chính nó?